In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# T46单次终态反馈与LOCAL对照（准备版）

仅CPU/synthetic/static验证，不曾本地运行真实模型/GPU。2原dev内容×OFF/LOCAL_A/B/TERMINAL_FEEDBACK_A/B=10视频40encode。
不改成功LAST检测A。固定T46一次更新，载体/state×polarity×sync/margin1/接收器均冻结，无择优时刻或强度扫描。

从完整OFF历史的T46独立no_grad preview到终态，只在detached终态latent上求既有tube hinge loss梯度。
用identity近似把终态下降方向经u控制映射到velocity delta_v=-u/sigma，再真实native更新/自由tail47..49。
LOCAL沿当前clean完整projection残差。所有方向先单位support RMS，再一次native probe匹配同消息未来LAST49 actualD预算。
LAST仅新生成诊断oracle，不是媒体arm/在线可用预算。preview与每个正式arm均deepcopy完整scheduler history。

主要预测=-g_terminal dot actual_D（next-state→terminal identity假设），不是-g dot u，也不是真实终态改善。
实际终态loss改善、相对LOCAL改善、MP4盲间隔相对LOCAL/OFF分别报告，固定4个case/message配对。
Guidance Watermarking只借鉴终态反馈/identity transport；不复现其VAE decoder/logcos/EOT/PCGrad方法。
无完整Transformer尾反传、无VAE梯度、无后验terminal筛选；失败保留10/40分母。

全轮260TF/144native（包含8preview native）/8unit probes/4terminal leaf backward/10decode/10MP4save/40encode。
主结果看paired_summary和mechanism_summary，质量PSNR/时间残差仅相对OFF；请人工检查内容、伪影、运动。
OFF排名不是FPR，预测下降不等于trajectory success。本轮复用开发内容，不是独立holdout。
新输出FlowTubeTerminalFeedback，保存预览、原history、梯度/控制/终态、媒体和盲读记录。
源码SHA：fa86224d500381986042da11409fcc5b79aa2361。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.metadata, json, os, signal, subprocess, sys
SOURCE_COMMIT = 'fa86224d500381986042da11409fcc5b79aa2361'
if SOURCE_COMMIT is None:
    raise RuntimeError('Pending source publication: rebuild with the published full SHA')
SOURCE_URL = 'https://github.com/RICHAAARC/SC-SSTW.git'
RUN_ID = 'flow_tube_terminal_feedback_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SOURCE = Path('/content') / (RUN_ID + '_source')
subprocess.run(['git', 'init', str(SOURCE)], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'remote', 'add', 'origin', SOURCE_URL], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', '--depth', '1', 'origin', SOURCE_COMMIT], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', SOURCE_COMMIT], check=True)
if subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip() != SOURCE_COMMIT:
    raise RuntimeError('Source checkout differs from pinned SHA')
def version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None
if version('torch') != '2.11.0+cu128':
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.11.0', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'diffusers==0.40.0', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
subprocess.run([sys.executable, '-c', "import torch,diffusers; assert str(torch.__version__) == '2.11.0+cu128', torch.__version__; assert diffusers.__version__ == '0.40.0', diffusers.__version__"], check=True)


In [ ]:
OUTPUT = Path('/content/drive/MyDrive/Video-WM/FlowTubeTerminalFeedback') / RUN_ID
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
ARCHIVE = OUTPUT.parent / (RUN_ID + '.source.zip')
subprocess.run(['git', '-C', str(SOURCE), 'archive', '--format=zip', '--output', str(ARCHIVE), SOURCE_COMMIT], check=True)
LOG = OUTPUT.parent / (RUN_ID + '.launcher.log')
command = [sys.executable, '-u', '-m', 'experiments.wan_state_clock.flow_tube_terminal_feedback_run', '--output', str(OUTPUT)]
with LOG.open('w') as log:
    process = subprocess.Popen(command, cwd=SOURCE, start_new_session=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end=''); log.write(line); log.flush()
        code = process.wait()
    except BaseException:
        try:
            os.killpg(process.pid, signal.SIGTERM); process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL); process.wait()
        except ProcessLookupError:
            pass
        raise
print('Output:', OUTPUT, 'Launcher log:', LOG, 'Source archive:', ARCHIVE)
if (OUTPUT / 'result.json').exists():
    result = json.loads((OUTPUT / 'result.json').read_text())
    print(json.dumps({k:result.get(k) for k in ('status','video_denominator','receiver_encode_denominator','fixed_calls','actual_calls_observed','recovery_summary','equivalence_summary','mechanism_summary','paired_summary','quality_summary','OFF_summary')}, indent=2))
if code:
    raise subprocess.CalledProcessError(code, command)
